In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# === STEP 1: Load Your Dataset ===
df = pd.read_excel("/content/clean_health_tips_no_vague.xlsx")  # Update path as needed

# === STEP 2: Preprocess ===
# Label encode mood
le_mood = LabelEncoder()
df["Mood_Encoded"] = le_mood.fit_transform(df["Mood"])

# Select features to use
features = ['Mood_Encoded', 'Calories', 'Proteins', 'Carbs', 'Fats',
            'Water (ml)', 'Heart Rate (bpm)', 'Steps', 'Sleep (min)']

# Normalize features
scaler = MinMaxScaler()
tip_vectors = scaler.fit_transform(df[features])

# === STEP 3: Function to Predict Best Tip ===
def predict_health_tip(user_input_dict):
    # Convert input to vector
    input_vector = pd.DataFrame([{
        'Mood_Encoded': le_mood.transform([user_input_dict['Mood']])[0],
        'Calories': user_input_dict['Calories'],
        'Proteins': user_input_dict['Proteins'],
        'Carbs': user_input_dict['Carbs'],
        'Fats': user_input_dict['Fats'],
        'Water (ml)': user_input_dict['Water (ml)'],
        'Heart Rate (bpm)': user_input_dict['Heart Rate (bpm)'],
        'Steps': user_input_dict['Steps'],
        'Sleep (min)': user_input_dict['Sleep (min)']
    }])

    # Normalize input
    input_vector_scaled = scaler.transform(input_vector)

    # Cosine similarity with all tips
    similarities = cosine_similarity(input_vector_scaled, tip_vectors)
    best_index = np.argmax(similarities)
    best_tip = df.loc[best_index, "Health Tip"]
    similarity_score = similarities[0][best_index]

    return {
        "tip_index": best_index,
        "similarity_score": round(similarity_score, 4),
        "predicted_tip": best_tip
    }

# === STEP 4: Example Usage ===
example_input = {
    'Mood': 'stressed',
    'Calories': 2500,
    'Proteins': 40,
    'Carbs': 300,
    'Fats': 80,
    'Water (ml)': 800,
    'Heart Rate (bpm)': 110,
    'Steps': 1200,
    'Sleep (min)': 320
}

result = predict_health_tip(example_input)
print("Predicted Tip:", result["predicted_tip"])
print("Similarity Score:", result["similarity_score"])
print("Tip Index:", result["tip_index"])


Predicted Tip: Elevated heart rate — focus on calming and hydration.
Similarity Score: 0.9679
Tip Index: 2553
